# Data Pipeline & Projection Geometry

This notebook walks through:
1. Loading a nuScenes sample and inspecting image/radar shapes
2. Projecting the BEV grid onto `CAM_FRONT` — visualizing which cells are visible
3. Radar BEV rasterization — heatmap of point density and speed
4. Ground-truth box distribution across the validation set

In [ ]:
import sys
import os
import yaml
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

CONFIG_PATH = os.path.join(PROJECT_ROOT, 'config.yaml')
with open(CONFIG_PATH, 'r') as f:
    cfg = yaml.safe_load(f)

NUSCENES_ROOT = cfg.get('data', {}).get('root', '/data/nuscenes')
VERSION       = cfg.get('data', {}).get('version', 'v1.0-mini')

print(f'nuScenes root : {NUSCENES_ROOT}')
print(f'Version       : {VERSION}')

try:
    from src.data.nuscenes_loader import NuScenesLoader  # type: ignore
    loader = NuScenesLoader(cfg)
    sample = loader.get_sample(0)

    print('\n=== Sample 0 ===')
    for cam_name, img in sample['images'].items():
        print(f'  {cam_name:<20}: shape {img.shape}')
    radar_pts = sample.get('radar_points', np.zeros((0, 5)))
    print(f'  Radar points          : {radar_pts.shape[0]} pts  (cols: x, y, z, vx, vy)')
    boxes = sample.get('gt_boxes', [])
    print(f'  GT boxes              : {len(boxes)}')

except Exception as e:
    print(f'\nLoader not available ({e}). Using synthetic placeholders.')

    CAMERAS = ['CAM_FRONT', 'CAM_FRONT_LEFT', 'CAM_FRONT_RIGHT',
               'CAM_BACK', 'CAM_BACK_LEFT', 'CAM_BACK_RIGHT']
    IMG_H, IMG_W = 900, 1600
    sample = {
        'images': {cam: np.random.randint(60, 180, (IMG_H, IMG_W, 3), dtype=np.uint8)
                   for cam in CAMERAS},
        'radar_points': np.column_stack([
            np.random.uniform(-50, 50,  300),   # x
            np.random.uniform(-10, 70,  300),   # y  (mostly forward)
            np.zeros(300),                      # z
            np.random.uniform(-15, 15, 300),    # vx
            np.random.uniform(-20, 20, 300),    # vy
        ]),
        'gt_boxes': [
            {'center': np.array([np.random.uniform(-40,40),
                                 np.random.uniform(-10,60), 0.5]),
             'class':  np.random.choice(['vehicle','pedestrian','cyclist'])
            } for _ in range(40)
        ],
    }

    print('\n=== Synthetic Sample ===')
    for cam_name, img in sample['images'].items():
        print(f'  {cam_name:<20}: shape {img.shape}')
    rp = sample['radar_points']
    print(f'  Radar points          : {rp.shape[0]} pts  (cols: x, y, z, vx, vy)')
    print(f'  GT boxes              : {len(sample["gt_boxes"])}')

## Camera Projection: BEV → Image

For each BEV reference point $(x, y, z_k)$ we compute the normalized image coordinate:

$$\begin{pmatrix} u \\ v \\ 1 \end{pmatrix} = \mathbf{K} [\mathbf{R}|\mathbf{t}] \begin{pmatrix} x \\ y \\ z_k \\ 1 \end{pmatrix}$$

Points with depth $> 0$ and $(u,v)$ inside the image boundary contribute attended features  
to the corresponding BEV query through the deformable attention mechanism.

In [ ]:
try:
    from src.geometry.projections import project_bev_to_image, build_bev_ref_points  # type: ignore
    HAVE_GEO = True
except ImportError:
    HAVE_GEO = False
    def project_bev_to_image(bev_pts, K, RT):
        n = bev_pts.shape[0]
        pts_h = np.hstack([bev_pts, np.ones((n, 1))])
        cam = (RT @ pts_h.T).T[:, :3]
        depth = cam[:, 2]
        valid = depth > 0.1
        uv = np.zeros((n, 2))
        if valid.sum() > 0:
            proj = (K @ cam[valid].T).T
            uv[valid] = proj[:, :2] / proj[:, 2:3]
        return uv, valid

bev_cfg = cfg.get('bev', {})
H_bev = bev_cfg.get('bev_h', 200)
W_bev = bev_cfg.get('bev_w', 200)
cell  = bev_cfg.get('cell_size', 0.5)
stride = 5

xs = np.linspace(-W_bev*cell/2 + cell/2, W_bev*cell/2 - cell/2, W_bev)[::stride]
ys = np.linspace(-H_bev*cell/2 + cell/2, H_bev*cell/2 - cell/2, H_bev)[::stride]
gx, gy = np.meshgrid(xs, ys)
bev_pts_flat = np.stack([gx.ravel(), gy.ravel(), np.zeros(gx.size)], axis=1)

# Use first real camera K/RT if available, else dummy CAM_FRONT
cam_front_img = list(sample['images'].values())[0]
IMG_H, IMG_W  = cam_front_img.shape[:2]

try:
    K   = loader.get_camera_intrinsics('CAM_FRONT', 0)    # type: ignore
    RT  = loader.get_camera_extrinsics('CAM_FRONT', 0)    # type: ignore
except Exception:
    # Fallback dummy: standard nuScenes-like front camera
    fx = 1266.0; fy = 1266.0; cx = IMG_W / 2; cy = IMG_H / 2
    K = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]], dtype=np.float64)
    R = np.array([[ 1,  0,  0],
                  [ 0,  0,  1],
                  [ 0, -1,  0]], dtype=np.float64)
    t_ego = np.array([0.0, -1.5, 0.9])   # cam pos in ego
    RT = np.eye(4, dtype=np.float64)
    RT[:3, :3] = R
    RT[:3,  3] = -R @ t_ego

uv, valid = project_bev_to_image(bev_pts_flat, K, RT)
in_img = (valid &
          (uv[:, 0] >= 0) & (uv[:, 0] < IMG_W) &
          (uv[:, 1] >= 0) & (uv[:, 1] < IMG_H))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: BEV coloured by visibility
ax = axes[0]
color_map = np.where(in_img.ravel(), 'lime',
            np.where(valid.ravel(), 'orange', 'dimgray'))
ax.scatter(bev_pts_flat[:, 0], bev_pts_flat[:, 1],
           c=color_map, s=6, alpha=0.7)
ax.set_aspect('equal')
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m) forward')
ax.set_title('BEV Grid: Visibility in CAM_FRONT')
legend_elems = [
    mpatches.Patch(color='lime',    label='Visible in image'),
    mpatches.Patch(color='orange',  label='In front, off-screen'),
    mpatches.Patch(color='dimgray', label='Behind camera'),
]
ax.legend(handles=legend_elems, fontsize=8, loc='lower right')
ax.plot(0, 0, 'r^', ms=10)

# Right: Image with projected BEV points overlaid
ax2 = axes[1]
ax2.imshow(cam_front_img)
if in_img.sum() > 0:
    dist2 = np.sqrt(bev_pts_flat[in_img, 0]**2 + bev_pts_flat[in_img, 1]**2)
    sc = ax2.scatter(uv[in_img, 0], uv[in_img, 1],
                     c=dist2, cmap='plasma', s=12, alpha=0.8, zorder=3)
    plt.colorbar(sc, ax=ax2, label='BEV dist from ego (m)')
ax2.set_title('BEV Reference Points → CAM_FRONT')
ax2.set_xlabel('u (px)'); ax2.set_ylabel('v (px)')

plt.tight_layout()
plt.show()
print(f'BEV cells sampled: {bev_pts_flat.shape[0]} | Visible in CAM_FRONT: {in_img.sum()} ({100*in_img.mean():.1f}%)')

## Radar BEV Rasterization

Radar returns are sparse — typically 50–300 points per sweep — but each point carries  
**Doppler velocity** $(v_x, v_y)$ which cameras cannot directly observe.  

The heatmap below shows:
- **Left**: point density per BEV cell (log scale)
- **Right**: mean speed magnitude $|v| = \sqrt{v_x^2 + v_y^2}$ per occupied cell

In [ ]:
radar_pts = sample['radar_points']   # (N, 5): x, y, z, vx, vy
rx, ry    = radar_pts[:, 0], radar_pts[:, 1]
speed     = np.sqrt(radar_pts[:, 3]**2 + radar_pts[:, 4]**2)

# BEV grid extent
bev_range = H_bev * cell / 2   # e.g. 50 m
bins = np.linspace(-bev_range, bev_range, H_bev + 1)

density, xedges, yedges = np.histogram2d(rx, ry, bins=[bins, bins])

# Speed heatmap: sum speed per cell, then divide by count
speed_sum, _, _ = np.histogram2d(rx, ry, bins=[bins, bins], weights=speed)
with np.errstate(invalid='ignore'):
    speed_map = np.where(density > 0, speed_sum / density, np.nan)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

def plot_bev_heatmap(ax, data, title, cmap, label, log_scale=False):
    plot_data = np.log1p(data) if log_scale else data
    im = ax.imshow(plot_data.T, origin='lower',
                   extent=[-bev_range, bev_range, -bev_range, bev_range],
                   cmap=cmap, interpolation='nearest', aspect='equal')
    plt.colorbar(im, ax=ax, label=label)
    ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m) forward')
    ax.set_title(title)
    ax.axhline(0, color='white', lw=0.5, ls='--', alpha=0.5)
    ax.axvline(0, color='white', lw=0.5, ls='--', alpha=0.5)
    # Scatter raw points
    ax.scatter(rx, ry, c='white', s=4, alpha=0.4, zorder=3)
    ax.plot(0, 0, 'r^', ms=9, zorder=5, label='Ego')
    ax.legend(fontsize=8, loc='upper right')

plot_bev_heatmap(axes[0], density,
                 f'Radar Point Density (N={radar_pts.shape[0]})',
                 'YlOrRd', 'log(1 + count)', log_scale=True)
plot_bev_heatmap(axes[1], speed_map,
                 'Mean Speed per BEV Cell',
                 'cool', '|v| (m/s)', log_scale=False)

plt.tight_layout()
plt.show()

print(f'Radar points  : {radar_pts.shape[0]}')
print(f'Occupied cells: {int((density > 0).sum())} / {H_bev * W_bev}')
print(f'Speed range   : {speed.min():.1f} – {speed.max():.1f} m/s  (mean {speed.mean():.1f} m/s)')

## GT Box Distribution

Understanding the spatial distribution of ground-truth annotations is critical for:
- Diagnosing class imbalance (pedestrians are far rarer than vehicles)
- Identifying detector blind spots (sparse coverage at range extremes)
- Tuning anchor / query point density

In [ ]:
CLASS_COLORS = {
    'vehicle':    '#4C9BE8',
    'pedestrian': '#5EC97E',
    'cyclist':    '#F4A23A',
}

# Try to iterate over all val samples; fall back to the single loaded sample
all_boxes = []

try:
    val_loader = NuScenesLoader(cfg, split='val')   # type: ignore
    n_samples = min(len(val_loader), 200)           # cap at 200 for speed
    for i in range(n_samples):
        s = val_loader.get_sample(i)
        all_boxes.extend(s.get('gt_boxes', []))
    print(f'Loaded {n_samples} val samples, {len(all_boxes)} total GT boxes.')
except Exception:
    # Synthetic fallback: 600 boxes across 3 classes
    rng = np.random.default_rng(42)
    for cls, n_cls in [('vehicle', 350), ('pedestrian', 180), ('cyclist', 70)]:
        for _ in range(n_cls):
            all_boxes.append({
                'center': np.array([rng.uniform(-45, 45),
                                    rng.uniform(-20, 60),
                                    rng.uniform(0, 2)]),
                'class': cls,
            })
    print(f'Using synthetic data: {len(all_boxes)} boxes.')

# Separate by class
by_class = {cls: [] for cls in CLASS_COLORS}
for box in all_boxes:
    c = box.get('class', 'vehicle')
    # Normalise class names from nuScenes taxonomy
    if 'vehicle' in c or 'car' in c or 'truck' in c or 'bus' in c:
        key = 'vehicle'
    elif 'ped' in c or 'person' in c:
        key = 'pedestrian'
    elif 'cycle' in c or 'bike' in c or 'motor' in c:
        key = 'cyclist'
    else:
        key = 'vehicle'   # lump remainder
    by_class[key].append(box['center'][:2])   # (x, y)

fig, ax = plt.subplots(figsize=(8, 8))

# Background density (all classes)
all_xy = np.array([b['center'][:2] for b in all_boxes])
bev_range = H_bev * cell / 2
if all_xy.shape[0] > 0:
    dens, xe, ye = np.histogram2d(all_xy[:, 0], all_xy[:, 1],
                                  bins=50,
                                  range=[[-bev_range, bev_range],
                                         [-bev_range, bev_range]])
    ax.imshow(np.log1p(dens.T), origin='lower',
              extent=[-bev_range, bev_range, -bev_range, bev_range],
              cmap='Greys', alpha=0.35, aspect='equal')

for cls, color in CLASS_COLORS.items():
    pts = np.array(by_class[cls]) if by_class[cls] else np.zeros((0, 2))
    ax.scatter(pts[:, 0] if pts.size else [],
               pts[:, 1] if pts.size else [],
               c=color, s=12, alpha=0.6,
               label=f'{cls} (n={len(by_class[cls])})',
               linewidths=0)

ax.set_xlim(-bev_range, bev_range)
ax.set_ylim(-bev_range, bev_range)
ax.set_aspect('equal')
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m) forward')
ax.set_title('GT Box Distribution in BEV (val set)')
ax.axhline(0, color='gray', lw=0.5, ls='--')
ax.axvline(0, color='gray', lw=0.5, ls='--')
ax.plot(0, 0, 'w^', ms=11, zorder=6, label='Ego')
ax.legend(fontsize=9, loc='upper right',
          facecolor='#1a1a1a', labelcolor='white',
          edgecolor='gray')

plt.tight_layout()
plt.show()

print('\nClass counts:')
for cls, pts in by_class.items():
    print(f'  {cls:<12}: {len(pts):>5}')